In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------------------------------------------
# 1. Load all sweep metrics
# ---------------------------------------------------------------
sweep_dir = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_per_segment_analysis/embedding_param_sweep")
segment_names = ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]

all_metrics = pd.concat([
    pd.read_parquet(sweep_dir / f"sweep_metrics_{seg}.parquet")
    for seg in segment_names
], ignore_index=True)

all_metrics["combined"] = all_metrics["spearman"] + all_metrics["trustworthiness_k15"]
print(f"Loaded {len(all_metrics)} total runs across {all_metrics['segment'].nunique()} segments")
all_metrics

In [ ]:
# ---------------------------------------------------------------
# 2. Spearman vs Trustworthiness scatter — per method, all segments
# ---------------------------------------------------------------
methods = ["tsne", "umap", "phate"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, method in zip(axes, methods):
    subset = all_metrics[all_metrics["method"] == method]
    for seg in segment_names:
        seg_data = subset[subset["segment"] == seg]
        ax.scatter(seg_data["spearman"], seg_data["trustworthiness_k15"],
                   label=seg, s=30, alpha=0.7)
    ax.set_xlabel("Spearman correlation")
    ax.set_ylabel("Trustworthiness (k=15)")
    ax.set_title(method.upper())
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.suptitle("Spearman vs Trustworthiness — all parameter combos", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 3. Heatmaps: metrics across parameter grid per method
#    (averaged across segments)
# ---------------------------------------------------------------

# --- t-SNE heatmap ---
tsne_data = all_metrics[all_metrics["method"] == "tsne"].copy()
tsne_pivot_sp = tsne_data.pivot_table(index="perplexity", columns="learning_rate",
                                       values="spearman", aggfunc="mean")
tsne_pivot_tr = tsne_data.pivot_table(index="perplexity", columns="learning_rate",
                                       values="trustworthiness_k15", aggfunc="mean")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(tsne_pivot_sp.values, aspect="auto", cmap="viridis")
axes[0].set_xticks(range(len(tsne_pivot_sp.columns)))
axes[0].set_xticklabels(tsne_pivot_sp.columns.astype(int))
axes[0].set_yticks(range(len(tsne_pivot_sp.index)))
axes[0].set_yticklabels(tsne_pivot_sp.index.astype(int))
axes[0].set_xlabel("learning_rate")
axes[0].set_ylabel("perplexity")
axes[0].set_title("t-SNE — Spearman (mean across segments)")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(tsne_pivot_tr.values, aspect="auto", cmap="viridis")
axes[1].set_xticks(range(len(tsne_pivot_tr.columns)))
axes[1].set_xticklabels(tsne_pivot_tr.columns.astype(int))
axes[1].set_yticks(range(len(tsne_pivot_tr.index)))
axes[1].set_yticklabels(tsne_pivot_tr.index.astype(int))
axes[1].set_xlabel("learning_rate")
axes[1].set_ylabel("perplexity")
axes[1].set_title("t-SNE — Trustworthiness k=15 (mean across segments)")
fig.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

# --- UMAP heatmap ---
umap_data = all_metrics[all_metrics["method"] == "umap"].copy()
umap_pivot_sp = umap_data.pivot_table(index="n_neighbors", columns="min_dist",
                                       values="spearman", aggfunc="mean")
umap_pivot_tr = umap_data.pivot_table(index="n_neighbors", columns="min_dist",
                                       values="trustworthiness_k15", aggfunc="mean")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(umap_pivot_sp.values, aspect="auto", cmap="viridis")
axes[0].set_xticks(range(len(umap_pivot_sp.columns)))
axes[0].set_xticklabels(umap_pivot_sp.columns)
axes[0].set_yticks(range(len(umap_pivot_sp.index)))
axes[0].set_yticklabels(umap_pivot_sp.index.astype(int))
axes[0].set_xlabel("min_dist")
axes[0].set_ylabel("n_neighbors")
axes[0].set_title("UMAP — Spearman (mean across segments)")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(umap_pivot_tr.values, aspect="auto", cmap="viridis")
axes[1].set_xticks(range(len(umap_pivot_tr.columns)))
axes[1].set_xticklabels(umap_pivot_tr.columns)
axes[1].set_yticks(range(len(umap_pivot_tr.index)))
axes[1].set_yticklabels(umap_pivot_tr.index.astype(int))
axes[1].set_xlabel("min_dist")
axes[1].set_ylabel("n_neighbors")
axes[1].set_title("UMAP — Trustworthiness k=15 (mean across segments)")
fig.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

# --- PHATE heatmap ---
phate_data = all_metrics[all_metrics["method"] == "phate"].copy()
phate_pivot_sp = phate_data.pivot_table(index="knn", columns="decay",
                                         values="spearman", aggfunc="mean")
phate_pivot_tr = phate_data.pivot_table(index="knn", columns="decay",
                                         values="trustworthiness_k15", aggfunc="mean")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(phate_pivot_sp.values, aspect="auto", cmap="viridis")
axes[0].set_xticks(range(len(phate_pivot_sp.columns)))
axes[0].set_xticklabels(phate_pivot_sp.columns.astype(int))
axes[0].set_yticks(range(len(phate_pivot_sp.index)))
axes[0].set_yticklabels(phate_pivot_sp.index.astype(int))
axes[0].set_xlabel("decay")
axes[0].set_ylabel("knn")
axes[0].set_title("PHATE — Spearman (mean across segments)")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(phate_pivot_tr.values, aspect="auto", cmap="viridis")
axes[1].set_xticks(range(len(phate_pivot_tr.columns)))
axes[1].set_xticklabels(phate_pivot_tr.columns.astype(int))
axes[1].set_yticks(range(len(phate_pivot_tr.index)))
axes[1].set_yticklabels(phate_pivot_tr.index.astype(int))
axes[1].set_xlabel("decay")
axes[1].set_ylabel("knn")
axes[1].set_title("PHATE — Trustworthiness k=15 (mean across segments)")
fig.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 4. Best params per method per segment
# ---------------------------------------------------------------
best_params = []
for method in ["tsne", "umap", "phate", "mds"]:
    subset = all_metrics[all_metrics["method"] == method].copy()
    for seg in segment_names:
        seg_sub = subset[subset["segment"] == seg].copy()
        if seg_sub.empty:
            continue
        best_idx = seg_sub["combined"].idxmax()
        best_params.append(seg_sub.loc[best_idx])

best_df = pd.DataFrame(best_params)
print("Best parameters per method per segment:\n")
for method in ["tsne", "umap", "phate", "mds"]:
    print(f"\n{'='*60}")
    print(f"{method.upper()}")
    print(f"{'='*60}")
    sub = best_df[best_df["method"] == method]
    cols = ["segment", "label", "spearman", "trustworthiness_k15", "combined"]
    print(sub[cols].to_string(index=False))

In [ ]:
# ---------------------------------------------------------------
# 5. Load and plot a specific embedding (pick from best_df)
# ---------------------------------------------------------------
def load_and_plot_embedding(seg_name, label, color_by="year"):
    """Load a saved embedding and plot it colored by metadata."""
    emb_dir = sweep_dir / "embeddings" / seg_name
    emb_df = pd.read_parquet(emb_dir / f"{label}.parquet")

    # Parse metadata
    split = emb_df["sample_id"].str.split("|")
    emb_df["country"]   = split.str[5]
    emb_df["continent"] = split.str[6]
    emb_df["year"]      = pd.to_numeric(split.str[7], errors="coerce")

    fig, ax = plt.subplots(figsize=(8, 6))
    if color_by == "year":
        sc = ax.scatter(emb_df["dim_1"], emb_df["dim_2"],
                        c=emb_df["year"], cmap="viridis", s=3, alpha=0.4, rasterized=True)
        fig.colorbar(sc, ax=ax, label="Year")
    else:
        cats = sorted(emb_df[color_by].dropna().unique())
        cmap = plt.get_cmap("tab10")
        color_map = {c: cmap(i / max(len(cats)-1, 1)) for i, c in enumerate(cats)}
        ax.scatter(emb_df["dim_1"], emb_df["dim_2"],
                   c=emb_df[color_by].map(color_map), s=3, alpha=0.4, rasterized=True)
        from matplotlib.lines import Line2D
        handles = [Line2D([0],[0], marker="o", color="w", markerfacecolor=color_map[c],
                          markersize=8, label=c) for c in cats]
        ax.legend(handles=handles, fontsize=20, title=color_by.capitalize())

    ax.set_title(f"{seg_name} — {label}")
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

# Example: plot the best t-SNE for HA colored by year
best_tsne_ha = best_df[(best_df["method"] == "tsne") & (best_df["segment"] == "HA")].iloc[0]
load_and_plot_embedding("HA", best_tsne_ha["label"], color_by="year")

In [ ]:
# ---------------------------------------------------------------
# 6. Compare top 3 per method for a given segment side-by-side
# ---------------------------------------------------------------
def plot_top_n(seg_name, method, n=3, color_by="year"):
    """Plot top N embeddings for a method/segment combo."""
    subset = all_metrics[(all_metrics["method"] == method) & 
                         (all_metrics["segment"] == seg_name)].copy()
    subset = subset.nlargest(n, "combined")

    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1:
        axes = [axes]

    emb_dir = sweep_dir / "embeddings" / seg_name
    for ax, (_, row) in zip(axes, subset.iterrows()):
        emb_df = pd.read_parquet(emb_dir / f"{row['label']}.parquet")
        split = emb_df["sample_id"].str.split("|")
        emb_df["year"] = pd.to_numeric(split.str[7], errors="coerce")
        emb_df["continent"] = split.str[6]

        if color_by == "year":
            sc = ax.scatter(emb_df["dim_1"], emb_df["dim_2"],
                            c=emb_df["year"], cmap="viridis", s=3, alpha=0.4, rasterized=True)
        else:
            cats = sorted(emb_df[color_by].dropna().unique())
            cmap_c = plt.get_cmap("tab10")
            cmap_dict = {c: cmap_c(i/max(len(cats)-1,1)) for i,c in enumerate(cats)}
            ax.scatter(emb_df["dim_1"], emb_df["dim_2"],
                       c=emb_df[color_by].map(cmap_dict), s=3, alpha=0.4, rasterized=True)

        ax.set_title(f"{row['label']}\nSp={row['spearman']:.4f} Tr={row['trustworthiness_k15']:.4f}",
                     fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

    plt.suptitle(f"{seg_name} — Top {n} {method.upper()} (by combined score)", fontsize=13)
    plt.tight_layout()
    plt.show()

# Example: top 3 UMAP for PB1
plot_top_n("PB1", "umap", n=3, color_by="year")

In [ ]:
# ---------------------------------------------------------------
# Heatmaps per segment (not averaged) — one figure per method
# ---------------------------------------------------------------

segment_names = ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]


import matplotlib.pyplot as plt

# Set globally before plotting
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

# --- t-SNE per segment ---
tsne_data = all_metrics[all_metrics["method"] == "tsne"].copy()

fig, axes = plt.subplots(2, len(segment_names), figsize=(4 * len(segment_names), 8))

for col, seg in enumerate(segment_names):
    seg_data = tsne_data[tsne_data["segment"] == seg]

    pivot_sp = seg_data.pivot_table(index="perplexity", columns="learning_rate",
                                     values="spearman")
    pivot_tr = seg_data.pivot_table(index="perplexity", columns="learning_rate",
                                     values="trustworthiness_k15")

    im0 = axes[0, col].imshow(pivot_sp.values, aspect="auto", cmap="viridis")
    axes[0, col].set_xticks(range(len(pivot_sp.columns)))
    axes[0, col].set_xticklabels(pivot_sp.columns.astype(int), rotation=45)
    axes[0, col].set_yticks(range(len(pivot_sp.index)))
    axes[0, col].set_yticklabels(pivot_sp.index.astype(int))
    axes[0, col].set_title(seg, fontweight="bold")
    if col == 0:
        axes[0, col].set_ylabel("perplexity\n(Spearman)", fontsize=9)
    fig.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)

    im1 = axes[1, col].imshow(pivot_tr.values, aspect="auto", cmap="viridis")
    axes[1, col].set_xticks(range(len(pivot_tr.columns)))
    axes[1, col].set_xticklabels(pivot_tr.columns.astype(int), fontsize=20, rotation=45)
    axes[1, col].set_yticks(range(len(pivot_tr.index)))
    axes[1, col].set_yticklabels(pivot_tr.index.astype(int), fontsize=20)
    if col == 0:
        axes[1, col].set_ylabel("perplexity\n(Trust k=15)", fontsize=9)
    axes[1, col].set_xlabel("learning_rate", fontsize=8)
    fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.suptitle("t-SNE — Spearman (top) & Trustworthiness (bottom) per segment", fontsize=35, y=1.01)
plt.tight_layout()
plt.show()


# --- UMAP per segment ---
umap_data = all_metrics[all_metrics["method"] == "umap"].copy()

fig, axes = plt.subplots(2, len(segment_names), figsize=(4 * len(segment_names), 8))

for col, seg in enumerate(segment_names):
    seg_data = umap_data[umap_data["segment"] == seg]

    pivot_sp = seg_data.pivot_table(index="n_neighbors", columns="min_dist",
                                     values="spearman")
    pivot_tr = seg_data.pivot_table(index="n_neighbors", columns="min_dist",
                                     values="trustworthiness_k15")

    im0 = axes[0, col].imshow(pivot_sp.values, aspect="auto", cmap="viridis")
    axes[0, col].set_xticks(range(len(pivot_sp.columns)))
    axes[0, col].set_xticklabels(pivot_sp.columns, fontsize=20, rotation=45)
    axes[0, col].set_yticks(range(len(pivot_sp.index)))
    axes[0, col].set_yticklabels(pivot_sp.index.astype(int), fontsize=20)
    axes[0, col].set_title(seg, fontsize=25, fontweight="bold")
    if col == 0:
        axes[0, col].set_ylabel("n_neighbors\n(Spearman)", fontsize=9)
    fig.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)

    im1 = axes[1, col].imshow(pivot_tr.values, aspect="auto", cmap="viridis")
    axes[1, col].set_xticks(range(len(pivot_tr.columns)))
    axes[1, col].set_xticklabels(pivot_tr.columns, fontsize=20, rotation=45)
    axes[1, col].set_yticks(range(len(pivot_tr.index)))
    axes[1, col].set_yticklabels(pivot_tr.index.astype(int), fontsize=20)
    if col == 0:
        axes[1, col].set_ylabel("n_neighbors\n(Trust k=15)", fontsize=9)
    axes[1, col].set_xlabel("min_dist", fontsize=8)
    fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.suptitle("UMAP — Spearman (top) & Trustworthiness (bottom) per segment", fontsize=35, y=1.01)
plt.tight_layout()
plt.show()


# --- PHATE per segment ---
phate_data = all_metrics[all_metrics["method"] == "phate"].copy()

fig, axes = plt.subplots(2, len(segment_names), figsize=(4 * len(segment_names), 8))

for col, seg in enumerate(segment_names):
    seg_data = phate_data[phate_data["segment"] == seg]

    pivot_sp = seg_data.pivot_table(index="knn", columns="decay",
                                     values="spearman")
    pivot_tr = seg_data.pivot_table(index="knn", columns="decay",
                                     values="trustworthiness_k15")

    im0 = axes[0, col].imshow(pivot_sp.values, aspect="auto", cmap="viridis")
    axes[0, col].set_xticks(range(len(pivot_sp.columns)))
    axes[0, col].set_xticklabels(pivot_sp.columns.astype(int), fontsize=15, rotation=45)
    axes[0, col].set_yticks(range(len(pivot_sp.index)))
    axes[0, col].set_yticklabels(pivot_sp.index.astype(int), fontsize=15)
    axes[0, col].set_title(seg, fontsize=22, fontweight="bold")
    if col == 0:
        axes[0, col].set_ylabel("knn\n(Spearman)", fontsize=22)
    cbar = fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=15)
    

    im1 = axes[1, col].imshow(pivot_tr.values, aspect="auto", cmap="viridis")
    axes[1, col].set_xticks(range(len(pivot_tr.columns)))
    axes[1, col].set_xticklabels(pivot_tr.columns.astype(int), fontsize=15, rotation=45)
    axes[1, col].set_yticks(range(len(pivot_tr.index)))
    axes[1, col].set_yticklabels(pivot_tr.index.astype(int), fontsize=15)
    if col == 0:
        axes[1, col].set_ylabel("knn\n(Trust k=15)", fontsize=22)
    axes[1, col].set_xlabel("decay", fontsize=22)
    #fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
    cbar = fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=15)

plt.suptitle("PHATE — Spearman (top) & Trustworthiness (bottom) per segment", fontsize=35, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 30,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
})

segment_names = ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]

# --- t-SNE per segment ---
tsne_data = all_metrics[all_metrics["method"] == "tsne"].copy()
fig, axes = plt.subplots(2, len(segment_names), figsize=(5 * len(segment_names), 10))

for col, seg in enumerate(segment_names):
    seg_data = tsne_data[tsne_data["segment"] == seg]
    pivot_sp = seg_data.pivot_table(index="perplexity", columns="learning_rate", values="spearman")
    pivot_tr = seg_data.pivot_table(index="perplexity", columns="learning_rate", values="trustworthiness_k15")

    im0 = axes[0, col].imshow(pivot_sp.values, aspect="auto", cmap="viridis")
    axes[0, col].set_xticks(range(len(pivot_sp.columns)))
    axes[0, col].set_xticklabels(pivot_sp.columns.astype(int), rotation=45)
    axes[0, col].set_yticks(range(len(pivot_sp.index)))
    axes[0, col].set_yticklabels(pivot_sp.index.astype(int))
    axes[0, col].set_title(seg, fontweight="bold")
    if col == 0:
        axes[0, col].set_ylabel("perplexity\n(Spearman)")
    cbar0 = fig.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)
    cbar0.ax.tick_params(labelsize=12)

    im1 = axes[1, col].imshow(pivot_tr.values, aspect="auto", cmap="viridis")
    axes[1, col].set_xticks(range(len(pivot_tr.columns)))
    axes[1, col].set_xticklabels(pivot_tr.columns.astype(int), rotation=45)
    axes[1, col].set_yticks(range(len(pivot_tr.index)))
    axes[1, col].set_yticklabels(pivot_tr.index.astype(int))
    if col == 0:
        axes[1, col].set_ylabel("perplexity\n(Trust k=15)")
    axes[1, col].set_xlabel("learning_rate")
    cbar1 = fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
    cbar1.ax.tick_params(labelsize=12)

plt.suptitle("t-SNE — Spearman (top) & Trustworthiness (bottom) per segment", fontsize=28, y=1.01)
plt.tight_layout()
plt.show()


# --- UMAP per segment ---
umap_data = all_metrics[all_metrics["method"] == "umap"].copy()
fig, axes = plt.subplots(2, len(segment_names), figsize=(5 * len(segment_names), 10))

for col, seg in enumerate(segment_names):
    seg_data = umap_data[umap_data["segment"] == seg]
    pivot_sp = seg_data.pivot_table(index="n_neighbors", columns="min_dist", values="spearman")
    pivot_tr = seg_data.pivot_table(index="n_neighbors", columns="min_dist", values="trustworthiness_k15")

    im0 = axes[0, col].imshow(pivot_sp.values, aspect="auto", cmap="viridis")
    axes[0, col].set_xticks(range(len(pivot_sp.columns)))
    axes[0, col].set_xticklabels(pivot_sp.columns, rotation=45)
    axes[0, col].set_yticks(range(len(pivot_sp.index)))
    axes[0, col].set_yticklabels(pivot_sp.index.astype(int))
    axes[0, col].set_title(seg, fontweight="bold")
    if col == 0:
        axes[0, col].set_ylabel("n_neighbors\n(Spearman)")
    cbar0 = fig.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)
    cbar0.ax.tick_params(labelsize=12)

    im1 = axes[1, col].imshow(pivot_tr.values, aspect="auto", cmap="viridis")
    axes[1, col].set_xticks(range(len(pivot_tr.columns)))
    axes[1, col].set_xticklabels(pivot_tr.columns, rotation=45)
    axes[1, col].set_yticks(range(len(pivot_tr.index)))
    axes[1, col].set_yticklabels(pivot_tr.index.astype(int))
    if col == 0:
        axes[1, col].set_ylabel("n_neighbors\n(Trust k=15)")
    axes[1, col].set_xlabel("min_dist")
    cbar1 = fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
    cbar1.ax.tick_params(labelsize=12)

plt.suptitle("UMAP — Spearman (top) & Trustworthiness (bottom) per segment", fontsize=28, y=1.01)
plt.tight_layout()
plt.show()


# --- PHATE per segment ---
phate_data = all_metrics[all_metrics["method"] == "phate"].copy()
fig, axes = plt.subplots(2, len(segment_names), figsize=(5 * len(segment_names), 10))

for col, seg in enumerate(segment_names):
    seg_data = phate_data[phate_data["segment"] == seg]
    pivot_sp = seg_data.pivot_table(index="knn", columns="decay", values="spearman")
    pivot_tr = seg_data.pivot_table(index="knn", columns="decay", values="trustworthiness_k15")

    im0 = axes[0, col].imshow(pivot_sp.values, aspect="auto", cmap="viridis")
    axes[0, col].set_xticks(range(len(pivot_sp.columns)))
    axes[0, col].set_xticklabels(pivot_sp.columns.astype(int), rotation=45)
    axes[0, col].set_yticks(range(len(pivot_sp.index)))
    axes[0, col].set_yticklabels(pivot_sp.index.astype(int))
    axes[0, col].set_title(seg, fontweight="bold")
    if col == 0:
        axes[0, col].set_ylabel("knn\n(Spearman)")
    cbar0 = fig.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)
    cbar0.ax.tick_params(labelsize=12)

    im1 = axes[1, col].imshow(pivot_tr.values, aspect="auto", cmap="viridis")
    axes[1, col].set_xticks(range(len(pivot_tr.columns)))
    axes[1, col].set_xticklabels(pivot_tr.columns.astype(int), rotation=45)
    axes[1, col].set_yticks(range(len(pivot_tr.index)))
    axes[1, col].set_yticklabels(pivot_tr.index.astype(int))
    if col == 0:
        axes[1, col].set_ylabel("knn\n(Trust k=15)")
    axes[1, col].set_xlabel("decay")
    cbar1 = fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
    cbar1.ax.tick_params(labelsize=12)

plt.suptitle("PHATE — Spearman (top) & Trustworthiness (bottom) per segment", fontsize=28, y=1.01)
plt.tight_layout()
plt.show()